# `pipeline.ipynb` - Orquestador del pipeline completo

Une todos los módulos anteriores dentro de un `try/except` general: si cualquier paso falla, se captura la excepción, se registra en el log con `exc_info=True` (traza completa), se cierra la auditoría con estado `ERROR` y observaciones, y se vuelve a lanzar la excepción. Si todo va bien, la auditoría se cierra con estado `OK` y la duración total.

> Depende de **todos** los módulos anteriores (`config`, `logging_config`, `transform`, `demografia`, `db`, `extract`, `consolidate`, `load`), así que debe ser el último `%run` de la cadena en `main.ipynb`. **No incluye celda de prueba aislada**: ejecutarlo aquí sería lanzar el pipeline completo dos veces (una al cargar este módulo y otra en `main.ipynb`, sección 6) — la ejecución real ya sirve como su propia prueba de integración.

In [ ]:
import logging
from datetime import datetime

## Orquestador

In [ ]:
def ejecutar_pipeline():
    """Ejecuta el pipeline ETL completo de principio a fin y devuelve (df_indicadores, df_cuarentena)."""
    fecha_inicio = datetime.now()
    logger.info("=" * 70)
    logger.info("INICIO DE EJECUCION DEL PIPELINE ETL - PROYECTO GEOSTAT")
    logger.info("=" * 70)

    id_ejecucion = registrar_inicio_ejecucion(fecha_inicio)
    logger.info(f"Ejecucion registrada en tb_ejecuciones_etl con id_ejecucion={id_ejecucion}")

    try:
        # 1) Extraccion demografica (API con fallback resiliente)
        poblacion, origen_poblacion = obtener_datos_demograficos()

        # 2) Extraccion + limpieza de la fuente Legacy (SQLite, por chunks)
        df_validos, df_cuarentena, total_leidos = extraer_y_limpiar_sqlite()

        # 3) Consolidacion estadistica (mediana) -> 1 fila por pais
        df_consolidado = consolidar_estadisticamente(df_validos)

        # 4) Calculo de indicadores (densidad, PIB per capita)
        df_indicadores = calcular_indicadores(df_consolidado, poblacion, origen_poblacion)

        # 5) Carga idempotente en PostgreSQL
        insertados = cargar_en_postgres(df_indicadores, df_cuarentena, id_ejecucion)

        fecha_fin = datetime.now()
        registrar_fin_ejecucion(
            id_ejecucion, fecha_inicio, fecha_fin, total_leidos, insertados,
            len(df_cuarentena), origen_poblacion, "OK",
            observaciones=f"Duracion: {(fecha_fin - fecha_inicio).total_seconds():.2f} s",
        )
        logger.info(f"EJECUCION FINALIZADA CORRECTAMENTE. Duracion: {(fecha_fin - fecha_inicio).total_seconds():.2f} s")
        logger.info("=" * 70)
        return df_indicadores, df_cuarentena

    except Exception as error:
        fecha_fin = datetime.now()
        registrar_fin_ejecucion(
            id_ejecucion, fecha_inicio, fecha_fin, 0, 0, 0, None, "ERROR",
            observaciones=str(error),
        )
        logger.error(f"EJECUCION FINALIZADA CON ERROR: {error}", exc_info=True)
        raise